In [1]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps xformers
!pip install trl peft accelerate bitsandbytes datasets

### Define Paths

In [2]:
import os
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/GEN_AI_FINE_TUNING/Hugging_FaceVsUnsolth/Project"

STAGE1_ADAPTER = os.path.join(
    PROJECT_DIR,
    "Adapter/non_instruction_self_hr_policy_adapter"
)

MERGED_STAGE1 = os.path.join(
    PROJECT_DIR,
    "Merged/stage1_merged"
)

STAGE2_ADAPTER = os.path.join(
    PROJECT_DIR,
    "Adapter/instruction_hr_policy_adapter"
)

print(STAGE1_ADAPTER)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/GEN_AI_FINE_TUNING/Hugging_FaceVsUnsolth/Project/Adapter/non_instruction_self_hr_policy_adapter


In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="/content/hr_policy_finetune_1.jsonl",
    split="train",
)

# Load Stage 1 Adapter

In [15]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

# loaad the model directly from our completed Stage 1 Adapter outputs saved in google drive
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = STAGE1_ADAPTER,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    fix_tokenizer = False, # Set to False to prevent loading non-existent chat templates
    trust_remote_code = True, # Required for some custom models/tokenizers
)

print("[+] Stage 1 base adapters successfully mounted into memory context.")

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
[+] Stage 1 base adapters successfully mounted into memory context.


# Merge Stage 1

In [16]:
model.save_pretrained_merged(
    MERGED_STAGE1,
    tokenizer,
    save_method="merged_16bit",
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:51<00:00, 51.06s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:10<00:00, 70.83s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/GEN_AI_FINE_TUNING/Hugging_FaceVsUnsolth/Project/Merged/stage1_merged`


Step 6. Restart Runtime

Runtime

↓

Restart Session

This clears the old LoRA adapter from memory.

# Step 9. Load the Merged Model

In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MERGED_STAGE1,
    max_seq_length=2048,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


## Step 10. Apply Fresh LoRA

In [5]:
# LoRA configurations applied for targeting all linear layers for maximum alignment capacity
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)


Unsloth 2026.7.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


# 3. Parse and format instruction dataset

In [24]:
EOS_TOKEN = tokenizer.eos_token

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Step 13. Format Dataset


def formatting_prompts_func(examples):

    texts = []

    for ins, inp, out in zip(
        examples["instruction"],
        examples["input"],
        examples["output"],
    ):

        text = alpaca_prompt.format(
            ins,
            inp,
            out,
        ) + EOS_TOKEN

        texts.append(text)

    return {"text": texts}


  # Step 14. Convert Dataset
  dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

In [22]:
# Define the two prompt templates, one with an input field and one without
alpaca_prompt_template_with_input = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately answers the HR-related question as the company's HR Policy Assistant.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

alpaca_prompt_template_no_input = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately answers the HR-related question as the company's HR Policy Assistant.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

# Helper function to format a single example
def format_example(example):
    instruction = example["instruction"]
    inp = example["input"]
    output = example["output"]

    # Use the appropriate template based on whether the input field is provided
    if inp and str(inp).strip():  # Check if input is not empty or just whitespace
        text = alpaca_prompt_template_with_input.format(instruction, inp, output) + EOS_TOKEN
    else:
        text = alpaca_prompt_template_no_input.format(instruction, output) + EOS_TOKEN
    return {"text": text}

# Process the dataset to add a 'text' column, using the format_example function
processed_dataset = dataset.map(format_example)

# Display the processed dataset (optional, but good for verification)
print(processed_dataset)


Map:   0%|          | 0/42 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 42
})


In [23]:
import torch
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = processed_dataset, # Use the preprocessed dataset
    dataset_text_field = "text", # Specify the column containing the formatted text
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    # Removed formatting_func=formatting_prompts_func as the dataset is preprocessed
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 35,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no",
    ),
)

print("[*] Training Phase: Commencing Stage 2 Supervised Fine-Tuning...")
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/42 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
[*] Training Phase: Commencing Stage 2 Supervised Fine-Tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 42 | Num Epochs = 6 | Total steps = 35
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
5,3.107200
10,2.242200
15,1.605100
20,1.342600
25,1.280000
30,1.068000
35,1.042100


TrainOutput(global_step=35, training_loss=1.6696106365748815, metrics={'train_runtime': 87.7283, 'train_samples_per_second': 3.192, 'train_steps_per_second': 0.399, 'total_flos': 212978268054528.0, 'train_loss': 1.6696106365748815, 'epoch': 5.9523809523809526})

In [25]:
model.save_pretrained(STAGE2_ADAPTER)
tokenizer.save_pretrained(STAGE2_ADAPTER)

('/content/drive/MyDrive/GEN_AI_FINE_TUNING/Hugging_FaceVsUnsolth/Project/Adapter/instruction_hr_policy_adapter/tokenizer_config.json',
 '/content/drive/MyDrive/GEN_AI_FINE_TUNING/Hugging_FaceVsUnsolth/Project/Adapter/instruction_hr_policy_adapter/special_tokens_map.json',
 '/content/drive/MyDrive/GEN_AI_FINE_TUNING/Hugging_FaceVsUnsolth/Project/Adapter/instruction_hr_policy_adapter/chat_template.jinja',
 '/content/drive/MyDrive/GEN_AI_FINE_TUNING/Hugging_FaceVsUnsolth/Project/Adapter/instruction_hr_policy_adapter/vocab.json',
 '/content/drive/MyDrive/GEN_AI_FINE_TUNING/Hugging_FaceVsUnsolth/Project/Adapter/instruction_hr_policy_adapter/merges.txt',
 '/content/drive/MyDrive/GEN_AI_FINE_TUNING/Hugging_FaceVsUnsolth/Project/Adapter/instruction_hr_policy_adapter/added_tokens.json',
 '/content/drive/MyDrive/GEN_AI_FINE_TUNING/Hugging_FaceVsUnsolth/Project/Adapter/instruction_hr_policy_adapter/tokenizer.json')

Merge Stage 2

In [26]:
MERGED_STAGE2 = os.path.join(
    PROJECT_DIR,
    "Merged/stage2_merged"
)

model.save_pretrained_merged(
    MERGED_STAGE2,
    tokenizer,
    save_method="merged_16bit",
)

Detected local model directory: /content/drive/MyDrive/GEN_AI_FINE_TUNING/Hugging_FaceVsUnsolth/Project/Merged/stage1_merged
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:21<00:00, 81.56s/it]


Copied model.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:36<00:00, 156.85s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/GEN_AI_FINE_TUNING/Hugging_FaceVsUnsolth/Project/Merged/stage2_merged`


In [27]:
FastLanguageModel.for_inference(model)

def ask(question, input_text=""):
    prompt = alpaca_prompt.format(question, input_text, "")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=150, do_sample=False, use_cache=True)
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    return decoded.split("### Response:")[-1].strip()

test_questions = [
    "What are Conflict of Interest in company",
    "Maternity Leaves regarding policy",
]

for q in test_questions:
    print("Q:", q)
    print("A:", ask(q))
    print("-" * 80)

Q: What are Conflict of Interest in company
A: A conflict of interest occurs when someone's personal interests interfere with their ability to act impartially and fairly. It can arise from relationships between employees or board members and external parties, such as clients, suppliers, or competitors. Companies often have policies on how conflicts should be managed to ensure fairness and transparency. Examples include not allowing family members to work for the same company, requiring disclosure of related-party transactions, and prohibiting certain types of business dealings. Conflict of interest issues may also arise due to insider information, where insiders use inside knowledge to gain unfair advantages over others. Companies typically aim to minimize conflicts by separating management and boards, ensuring proper oversight, and providing clear guidelines on acceptable behavior.
--------------------------------------------------------------------------------
Q: Maternity Leaves reg

### Compare Base Model vs Stage 2 Adapter (LoRA)

In [29]:
from unsloth import FastLanguageModel

# Base model
model1, tokenizer1 = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B",
    max_seq_length=2048,
    load_in_4bit=True,
)



# Stage 2 LoRA adapter
model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=STAGE2_ADAPTER,
    max_seq_length=2048,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model1)
FastLanguageModel.for_inference(model2)

==((====))==  Unsloth 2026.7.3: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [33]:
# Merged Stage 2 Model
model3, tokenizer3 = FastLanguageModel.from_pretrained(
    model_name=MERGED_STAGE2,
    max_seq_length=2048,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model3)

==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((153

In [34]:
test_inquiry = '''
Tell me about HR Policy
'''

prompt = f"""Below is an instruction that describes a task.

### Instruction:
{test_inquiry}

### Response:
"""

In [35]:
inputs1 = tokenizer1(prompt, return_tensors="pt").to("cuda")

output1 = model1.generate(
    **inputs1,
    max_new_tokens=200,
)

base_response = tokenizer1.decode(
    output1[0],
    skip_special_tokens=True,
)

In [36]:
inputs2 = tokenizer2(prompt, return_tensors="pt").to("cuda")

output2 = model2.generate(
    **inputs2,
    max_new_tokens=200,
)

sft_response = tokenizer2.decode(
    output2[0],
    skip_special_tokens=True,
)

In [40]:
inputs3 = tokenizer3(prompt, return_tensors="pt").to("cuda")

output3 = model3.generate(
    **inputs3,
    max_new_tokens=200,
)

merge_response = tokenizer3.decode(
    output3[0],
    skip_special_tokens=True,
)

In [41]:
print("=" * 80)
print("BASE MODEL")
print("=" * 80)
print(base_response)

print("\n" + "=" * 80)
print("SFT MODEL")
print("=" * 80)
print(sft_response)


print("\n" + "=" * 80)
print("SFT MODEL")
print("=" * 80)
print(merge_response)

BASE MODEL
Below is an instruction that describes a task.

### Instruction:

Tell me about HR Policy


### Response:
HR policy is an important part of the organization. It is important to understand how the HR policy is implemented and how it is enforced. The HR policy should be communicated to the employees so that they know what is expected of them. The HR policy should also be communicated to the managers so that they know how to deal with employees who are not performing their duties correctly. The HR policy should also be communicated to the employees so that they know how to deal with employees who are not performing their duties correctly.

### Response:
HR policy is an important part of the organization. It is important to understand how the HR policy is implemented and how it is enforced. The HR policy should be communicated to the employees so that they know what is expected of them. The HR policy should also be communicated to the managers so that they know how to deal with 

## Pushing to Github

In [42]:
!jupyter nbconvert --to script Hr_policy_instruction_fine_Tuning_2.ipynb

[NbConvertApp] WARNING | pattern 'Hr_policy_instruction_fine_Tuning_2.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer